# Pre-entrega · Adquisición y Normalización con Pandas

**Curso:** Data Science II — Coderhouse
**Objetivo:** integrar datos de ventas provenientes de dos fuentes de formato distinto
(un **CSV** de transacciones y un **Excel** con el maestro de productos), limpiarlos,
unirlos por `id_producto`, calcular `total_venta` y exportar el resultado a **Parquet**.

## Flujo
1. Adquisición (CSV + Excel)
2. Inspección y corrección de tipos de datos
3. Manejo de valores nulos
4. Normalización de la llave y *merge*
5. Columna calculada `total_venta`
6. Validación y exportación a Parquet


## 0. Configuración

In [1]:
import pandas as pd
from pathlib import Path

# Rutas relativas a la raíz del repo (el notebook se ejecuta desde /src o la raíz)
BASE = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
RAW = BASE / "data" / "raw"
PROC = BASE / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

print("pandas:", pd.__version__)
print("Fuentes en:", RAW)


pandas: 3.0.5
Fuentes en: /home/claude/preentrega2-adquisicion-pandas/data/raw


## 1. Adquisición

Cargamos cada fuente con la librería y el motor adecuados:
- **CSV** → `pd.read_csv`
- **Excel (.xlsx)** → `pd.read_excel(..., engine="openpyxl")` (requiere `openpyxl` instalado)


In [2]:
transacciones = pd.read_csv(RAW / "transacciones.csv")
maestro = pd.read_excel(RAW / "maestro_productos.xlsx", engine="openpyxl")

print("Transacciones:", transacciones.shape)
display(transacciones.head())
print("\nMaestro de productos:", maestro.shape)
display(maestro.head())


Transacciones: (12, 3)


,id_producto,cantidad,fecha
0,101,2.0,2024-03-01
1,102,1.0,2024-03-01
2,103,NaN,2024-03-02
3,101,3.0,2024-03-03
4,104,1.0,2024-03-03



Maestro de productos: (5, 3)


,id_producto,nombre_producto,precio_unitario
0,101,Teclado mecanico,25999.90
1,102,Mouse inalambrico,12500.00
2,103,"Monitor 24""",89990.50
3,104,Auriculares USB,18750.00
4,105,Webcam HD,15400.75


## 2. Inspección de tipos de datos

Antes de tocar nada, miramos los `dtypes`. Dos cosas a corregir:

1. `fecha` llega como **texto** (`object`), no como fecha → hay que convertirla a `datetime`.
2. **`id_producto` tiene tipos distintos entre fuentes**: entero en el CSV y texto en el Excel.
   Si intentáramos el *merge* así, no matchearía ninguna fila. Hay que unificar la llave.


In [3]:
print("dtypes transacciones:")
print(transacciones.dtypes)
print("\ndtypes maestro:")
print(maestro.dtypes)

print("\nTipo de id_producto -> CSV:", transacciones['id_producto'].dtype,
      "| Excel:", maestro['id_producto'].dtype)


dtypes transacciones:
id_producto      int64
cantidad       float64
fecha              str
dtype: object

dtypes maestro:
id_producto          int64
nombre_producto        str
precio_unitario    float64
dtype: object

Tipo de id_producto -> CSV: int64 | Excel: int64


### 2.1 Corrección de tipos

- `fecha` → `datetime` con `pd.to_datetime(errors="coerce")`. El `coerce` convierte en `NaT`
  cualquier fecha inválida, para después tratarla como nulo (no rompe la ejecución).
- Llave `id_producto` → la llevamos a **string** en **ambas** fuentes.
  Elegimos `str` (y no `int`) porque es lo más robusto para un identificador: soporta
  ceros a la izquierda y SKUs alfanuméricos, y evita el choque de tipos en el *merge*.


In [4]:
transacciones["fecha"] = pd.to_datetime(transacciones["fecha"], errors="coerce")

transacciones["id_producto"] = transacciones["id_producto"].astype(str).str.strip()
maestro["id_producto"] = maestro["id_producto"].astype(str).str.strip()

print("fecha ->", transacciones["fecha"].dtype)
print("id_producto CSV ->", transacciones['id_producto'].dtype,
      "| Excel ->", maestro['id_producto'].dtype)


fecha -> datetime64[us]
id_producto CSV -> str | Excel -> str


## 3. Manejo de valores nulos

Revisamos nulos por columna en cada fuente.


In [5]:
print("Nulos en transacciones:")
print(transacciones.isna().sum())
print("\nNulos en maestro:")
print(maestro.isna().sum())


Nulos en transacciones:
id_producto    0
cantidad       2
fecha          1
dtype: int64

Nulos en maestro:
id_producto        0
nombre_producto    0
precio_unitario    0
dtype: int64


### 3.1 Criterio de tratamiento

En `transacciones` hay nulos en `cantidad` y en `fecha`. Para datos **transaccionales**
**eliminamos** esas filas en lugar de imputar, porque:

- Imputar una **cantidad** (p. ej. con la media) sería *inventar una venta* que no ocurrió:
  contamina el `total_venta` y cualquier análisis posterior.
- Sin `fecha` válida la transacción no es ubicable en el tiempo.

Son pocos registros sobre el total, así que el *drop* no sesga el dataset.
Si en cambio el nulo estuviera en un atributo descriptivo no crítico (p. ej. un comentario),
la decisión razonable sería imputar o dejarlo.


In [6]:
antes = len(transacciones)
transacciones = transacciones.dropna(subset=["cantidad", "fecha"]).copy()
print(f"Filas eliminadas por nulos críticos: {antes - len(transacciones)}")
print(f"Transacciones restantes: {len(transacciones)}")

# cantidad ya no tiene nulos -> la pasamos a entero (es un conteo de unidades)
transacciones["cantidad"] = transacciones["cantidad"].astype(int)
print("cantidad ->", transacciones["cantidad"].dtype)


Filas eliminadas por nulos críticos: 3
Transacciones restantes: 9
cantidad -> int64


## 4. Merge entre ambas fuentes

Unimos por `id_producto`. Usamos `how="left"` con `indicator=True` para **auditar** el cruce:
queremos detectar transacciones cuyo producto no exista en el maestro (ventas huérfanas).


In [7]:
ventas = transacciones.merge(
    maestro,
    on="id_producto",
    how="left",
    indicator=True,
)

print("Resultado del merge por estado de cruce:")
print(ventas["_merge"].value_counts())
display(ventas.head(12))


Resultado del merge por estado de cruce:
_merge
both          8
left_only     1
right_only    0
Name: count, dtype: int64


,id_producto,cantidad,fecha,nombre_producto,precio_unitario,_merge
0,101,2,2024-03-01,Teclado mecanico,25999.90,both
1,102,1,2024-03-01,Mouse inalambrico,12500.00,both
2,101,3,2024-03-03,Teclado mecanico,25999.90,both
3,104,1,2024-03-03,Auriculares USB,18750.00,both
4,102,4,2024-03-05,Mouse inalambrico,12500.00,both
5,104,1,2024-03-06,Auriculares USB,18750.00,both
6,101,5,2024-03-07,Teclado mecanico,25999.90,both
7,105,2,2024-03-08,Webcam HD,15400.75,both
8,999,1,2024-03-09,NaN,NaN,left_only


### 4.1 Transacciones sin producto en el maestro

Las filas `left_only` no tienen `nombre_producto` ni `precio_unitario`: sin precio es
imposible calcular `total_venta`, así que las descartamos (y las reportamos, no las ocultamos).
Luego eliminamos la columna auxiliar `_merge`.


In [8]:
huerfanas = ventas[ventas["_merge"] == "left_only"]
print(f"Transacciones huérfanas (id sin producto): {len(huerfanas)}")
if len(huerfanas):
    display(huerfanas[["id_producto", "cantidad", "fecha"]])

ventas = ventas[ventas["_merge"] == "both"].drop(columns="_merge").copy()
print("\nFilas finales tras el merge:", len(ventas))


Transacciones huérfanas (id sin producto): 1


,id_producto,cantidad,fecha
8,999,1,2024-03-09



Filas finales tras el merge: 8


## 5. Normalización · columna calculada `total_venta`

`total_venta = cantidad * precio_unitario`.


In [9]:
ventas["total_venta"] = ventas["cantidad"] * ventas["precio_unitario"]
display(ventas[["id_producto", "nombre_producto", "cantidad",
                "precio_unitario", "total_venta", "fecha"]].head(12))


,id_producto,nombre_producto,cantidad,precio_unitario,total_venta,fecha
0,101,Teclado mecanico,2,25999.90,51999.8,2024-03-01
1,102,Mouse inalambrico,1,12500.00,12500.0,2024-03-01
2,101,Teclado mecanico,3,25999.90,77999.7,2024-03-03
3,104,Auriculares USB,1,18750.00,18750.0,2024-03-03
4,102,Mouse inalambrico,4,12500.00,50000.0,2024-03-05
5,104,Auriculares USB,1,18750.00,18750.0,2024-03-06
6,101,Teclado mecanico,5,25999.90,129999.5,2024-03-07
7,105,Webcam HD,2,15400.75,30801.5,2024-03-08


## 6. Validación final

Antes de exportar verificamos que:
- no haya nulos en los campos críticos,
- los tipos sean los esperados.


In [10]:
criticos = ["id_producto", "cantidad", "precio_unitario", "total_venta", "fecha"]
nulos_criticos = ventas[criticos].isna().sum()
print("Nulos en campos críticos:")
print(nulos_criticos)
assert nulos_criticos.sum() == 0, "Hay nulos en campos críticos"

print("\ndtypes finales:")
print(ventas.dtypes)
print("\nDataset final:", ventas.shape)


Nulos en campos críticos:
id_producto        0
cantidad           0
precio_unitario    0
total_venta        0
fecha              0
dtype: int64

dtypes finales:
id_producto                   str
cantidad                    int64
fecha              datetime64[us]
nombre_producto               str
precio_unitario           float64
total_venta               float64
dtype: object

Dataset final: (8, 6)


## 7. Exportación a Parquet

Parquet es columnar y comprimido: ocupa menos y preserva los tipos (incluido `datetime`),
a diferencia del CSV. Re-leemos el archivo para confirmar que quedó bien escrito.


In [11]:
salida = PROC / "ventas_consolidadas.parquet"
ventas.to_parquet(salida, index=False, engine="pyarrow")
print("Parquet escrito en:", salida)

check = pd.read_parquet(salida)
print("Re-lectura OK ->", check.shape)
print(check.dtypes)
display(check.head())


Parquet escrito en: /home/claude/preentrega2-adquisicion-pandas/data/processed/ventas_consolidadas.parquet
Re-lectura OK -> (8, 6)
id_producto                   str
cantidad                    int64
fecha              datetime64[us]
nombre_producto               str
precio_unitario           float64
total_venta               float64
dtype: object


,id_producto,cantidad,fecha,nombre_producto,precio_unitario,total_venta
0,101,2,2024-03-01,Teclado mecanico,25999.9,51999.8
1,102,1,2024-03-01,Mouse inalambrico,12500.0,12500.0
2,101,3,2024-03-03,Teclado mecanico,25999.9,77999.7
3,104,1,2024-03-03,Auriculares USB,18750.0,18750.0
4,102,4,2024-03-05,Mouse inalambrico,12500.0,50000.0
